In [1]:
%load_ext autoreload
%autoreload 2

import hydra

from genpp import BASE_DIR
from genpp.configs import register_resolvers

try:
    register_resolvers()
except ValueError:
    pass

# Test if Model works in general

In [2]:
with hydra.initialize_config_dir(version_base=None, config_dir=str(BASE_DIR / "configs")):
    cfg = hydra.compose(config_name="base_engression")

In [3]:
datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()
datamodule.setup("fit")
dataloader = datamodule.train_dataloader()

Configuration hash: 1333871877b7b5de25d7b65483ae3f7608524ce480ab5274278158e8a78c286c
Cached tensor data found. Verifying configuration...
Using cached tensor data from /home/feik/GenPP/src/genpp/data/weatherbench2/cache/tensor_1333871877b7b5de25d7b65483ae3f7608524ce480ab5274278158e8a78c286c.pt.


In [4]:
model = hydra.utils.instantiate(
    cfg.model,
    rescaler=datamodule.y_reverseModules if cfg.model.use_rescaler else None,
)

In [5]:
trainer = hydra.utils.instantiate(
    cfg.trainer,
    logger=None,
    detect_anomaly=True,
    accelerator="gpu",
    devices=[0],
    max_epochs=30,
    overfit_batches=1,
    check_val_every_n_epoch=3,
)
model.compile()
trainer.fit(model, datamodule=datamodule)

You have turned on `Trainer(detect_anomaly=True)`. This will significantly slow down compute speed and is recommended only for model debugging.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
`Trainer(overfit_batches=1)` was configured so 1 batch will be used.
You are using a CUDA device ('NVIDIA RTX A5000') that has Tensor Cores. To properly utilize them, you should set `torch.set_fl

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:252: You requested to overfit but enabled train dataloader shuffling. We are turning off the train dataloader shuffling for you.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=30` reached.


## Try to load the model from checkpoint

In [ ]:
import os
from pathlib import Path

cwd = Path(os.getcwd())
checkpoint_path = list((cwd / "lightning_logs" / "version_0" / "checkpoints").glob("*.ckpt"))[0]
checkpoint_path

PosixPath('/home/feik/GenPP/src/genpp/models/notebooks/lightning_logs/version_0/checkpoints/epoch=5-val_loss=10.25.ckpt')

In [7]:
for batch in dataloader:
    break

In [9]:
mod_new = model.__class__.load_from_checkpoint(checkpoint_path)

In [10]:
def move_to_device(batch, device):
    """
    Recursively moves all tensors inside a nested dict to the given device.
    """
    if isinstance(batch, dict):
        return {k: move_to_device(v, device) for k, v in batch.items()}

    # If it has .to(), treat it as a tensor-like object
    if hasattr(batch, "to"):
        return batch.to(device)

    return batch  # leave other objects as-is


batch = move_to_device(batch, mod_new.device)

In [12]:
res = mod_new.predict_step(batch)
print(res.shape)

torch.Size([8, 50, 2, 37, 31])


## Some random short tests

In [ ]:
import torch
from einops import rearrange

from genpp.models.scores import PatchwiseEnergyScore

In [ ]:
size = 30
channels = 1
beta = 1.0
t_c = torch.randint(0, 10, (64, 10, channels, size, size)) * 1.0
t_res = torch.randint(0, 10, (64, channels, size, size)) * 10.0

for p in range(3, 22, 2):
    pes = PatchwiseEnergyScore(height=size, width=size, patch_size=p, beta=beta)

    t_c_flat = rearrange(t_c, "b n c h w -> b n (c h w)")
    t_res_flat = rearrange(t_res, "b c h w -> b (c h w)")
    res = pes(t_c_flat, t_res_flat, mode="complete")
    res /= p**beta

    print(res.mean())